# Train COCO128 @320 (device+cuBLAS) → save best.pt → detect → show image
Trains device-resident, saves `best.pt`, then runs `yolo detect` and displays the annotated result.
**Runtime → GPU**, Run all. (Training ~34 min for 100 epochs; lower EPOCHS for a quicker demo.)


In [ ]:
!nvidia-smi -L


In [ ]:
%cd /content
!rm -rf yolov8_cpp
!git clone -q --branch main https://github.com/yomei-o/yolov8_cpp.git
%cd /content/yolov8_cpp
!wget -q https://github.com/ultralytics/yolov5/releases/download/v1.0/coco128.zip && unzip -q -o coco128.zip


### 1. Build (device + cuBLAS) and train — saves last.pt / best.pt each epoch


In [ ]:
!nvcc -x cu -O2 -std=c++17 --extended-lambda -arch=native -DUSE_CUDA -DUSE_CUBLAS -Ipure/third_party pure/dtrain_coco.cpp -lcublas -o dtrain_coco
EPOCHS=100  # <- lower this (e.g. 40) for a faster demo
!./dtrain_coco coco128/images/train2017 320 8 $EPOCHS


### 2. Build the detector CLI (plain C++/g++, no CUDA needed) and detect with best.pt


In [ ]:
!g++ -O2 -std=c++20 -Ipure/third_party pure/yolo.cpp -o yolo
IMG='coco128/images/train2017/000000000009.jpg'
!./yolo detect --weights best.pt --source $IMG --imgsz 320 --conf 0.25 --out det.png


### 3. Show the annotated result


In [ ]:
from IPython.display import Image, display
print('input:'); display(Image('coco128/images/train2017/000000000009.jpg', width=480))
print('detections (best.pt):'); display(Image('det.png', width=480))


---
COCO128 is 128 images so the model overfits its training set — detecting on a **training** image
should show sensible boxes. `best.pt` is a standard state_dict (loads in Ultralytics too).
